# Reformat Synthetic Data to Feather Format

This notebook converts synthetic data from .npy format to .feather format with proper column headers and label formatting to match the test_data.feather structure.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import os

# Define label mapping (0-7 to fuel types)
label_map = {
    0: 'DEB',
    1: 'DEM', 
    2: 'DMMP',
    3: 'DPM',
    4: 'DtBP',
    5: 'JP8',
    6: 'MES',
    7: 'TEPO'
}

## Load Reference Data Format

In [ ]:
# Load test_data.feather to understand the expected format
test_df = pd.read_feather('Data/test_data.feather')
print(f"Test data shape: {test_df.shape}")

# Get p_ and n_ columns
p_cols = [c for c in test_df.columns if c.startswith('p_')]
n_cols = [c for c in test_df.columns if c.startswith('n_')]

print(f"Number of p_ columns: {len(p_cols)} (from {p_cols[0]} to {p_cols[-1]})")
print(f"Number of n_ columns: {len(n_cols)} (from {n_cols[0]} to {n_cols[-1]})")
print(f"Label column values: {test_df['Label'].unique()}")
print(f"\nFirst few columns: {test_df.columns[:5].tolist()}")
print(f"Last few columns: {test_df.columns[-5:].tolist()}")

## Define Conversion Function

In [ ]:
def convert_spectra_to_feather(spectra, labels, std_value, output_dir='results'):
    """
    Convert synthetic spectra data to feather format with proper column headers and labels.
    
    Args:
        spectra: numpy array of shape (n_samples, 1676) with spectral data
        labels: numpy array of shape (n_samples,) with numeric labels 0-7
        std_value: float (1.0, 1.5, or 2.0) for file naming
        output_dir: directory to save output file
    
    Returns:
        DataFrame with proper formatting
    """
    # Split spectra into p_ and n_ columns (838 each)
    n_bins = 838
    start_bin = 184  # Bin numbering starts at 184
    
    # Create column names for p_ (positive ion mode)
    p_columns = [f'p_{i}' for i in range(start_bin, start_bin + n_bins)]
    # Create column names for n_ (negative ion mode)
    n_columns = [f'n_{i}' for i in range(start_bin, start_bin + n_bins)]
    
    # Split spectra data
    p_data = spectra[:, :n_bins]
    n_data = spectra[:, n_bins:]
    
    # Create DataFrame with index column first
    df = pd.DataFrame({'index': range(len(spectra))})
    
    # Add p_ columns
    df_p = pd.DataFrame(p_data, columns=p_columns)
    df = pd.concat([df, df_p], axis=1)
    
    # Add n_ columns
    df_n = pd.DataFrame(n_data, columns=n_columns)
    df = pd.concat([df, df_n], axis=1)
    
    # Convert numeric labels to string labels
    df['Label'] = [label_map[label] for label in labels]
    
    # Get unique labels for file naming
    unique_labels = sorted(df['Label'].unique())
    labels_str = '_'.join(unique_labels)
    
    # Create output filename
    output_path = os.path.join(output_dir, f'synthetic_spectra_std{std_value}_{labels_str}.feather')
    
    # Save as feather (index is already a column, so don't reset_index)
    df.to_feather(output_path)
    
    print(f"Saved {output_path}")
    print(f"  Shape: {df.shape}")
    print(f"  Labels: {unique_labels}")
    
    return df, output_path


def convert_latents_to_feather(latents, labels, std_value, output_dir='results'):
    """
    Convert synthetic latent data to feather format with proper column headers and labels.
    
    Args:
        latents: numpy array of shape (n_samples, 512) with latent data
        labels: numpy array of shape (n_samples,) with numeric labels 0-7
        std_value: float (1.0, 1.5, or 2.0) for file naming
        output_dir: directory to save output file
    
    Returns:
        DataFrame with proper formatting
    """
    # Create DataFrame with index column first
    df = pd.DataFrame({'index': range(len(latents))})
    
    # Create column names for latent dimensions
    n_latent_dims = latents.shape[1]
    latent_columns = [f'latent_{i}' for i in range(n_latent_dims)]
    
    # Add latent columns
    df_latents = pd.DataFrame(latents, columns=latent_columns)
    df = pd.concat([df, df_latents], axis=1)
    
    # Convert numeric labels to string labels
    df['Label'] = [label_map[label] for label in labels]
    
    # Get unique labels for file naming
    unique_labels = sorted(df['Label'].unique())
    labels_str = '_'.join(unique_labels)
    
    # Create output filename
    output_path = os.path.join(output_dir, f'synthetic_latents_std{std_value}_{labels_str}.feather')
    
    # Save as feather (index is already a column, so don't reset_index)
    df.to_feather(output_path)
    
    print(f"Saved {output_path}")
    print(f"  Shape: {df.shape}")
    print(f"  Labels: {unique_labels}")
    
    return df, output_path

## Process Standard Deviation 1.0 Data

In [ ]:
# Load std 1.0 data
labels_1_0 = np.load('results/generated_labels_std1.0.npy')
latents_1_0 = np.load('results/generated_latents_std1.0.npy')
spectra_1_0 = np.load('results/generated_spectra_std1.0.npy')

print(f"Std 1.0 - Labels shape: {labels_1_0.shape}")
print(f"         Latents shape: {latents_1_0.shape}")
print(f"         Spectra shape: {spectra_1_0.shape}")
print(f"Unique labels: {np.unique(labels_1_0)}")

# Convert to feather
df_spectra_1_0, path_spectra_1_0 = convert_spectra_to_feather(spectra_1_0, labels_1_0, 1.0)
df_latents_1_0, path_latents_1_0 = convert_latents_to_feather(latents_1_0, labels_1_0, 1.0)

## Process Standard Deviation 1.5 Data

In [ ]:
# Load std 1.5 data
labels_1_5 = np.load('results/generated_labels_std1.5.npy')
latents_1_5 = np.load('results/generated_latents_std1.5.npy')
spectra_1_5 = np.load('results/generated_spectra_std1.5.npy')

print(f"Std 1.5 - Labels shape: {labels_1_5.shape}")
print(f"         Latents shape: {latents_1_5.shape}")
print(f"         Spectra shape: {spectra_1_5.shape}")
print(f"Unique labels: {np.unique(labels_1_5)}")

# Convert to feather
df_spectra_1_5, path_spectra_1_5 = convert_spectra_to_feather(spectra_1_5, labels_1_5, 1.5)
df_latents_1_5, path_latents_1_5 = convert_latents_to_feather(latents_1_5, labels_1_5, 1.5)

## Process Standard Deviation 2.0 Data

In [ ]:
# Load std 2.0 data
labels_2_0 = np.load('results/generated_labels_std2.0.npy')
latents_2_0 = np.load('results/generated_latents_std2.0.npy')
spectra_2_0 = np.load('results/generated_spectra_std2.0.npy')

print(f"Std 2.0 - Labels shape: {labels_2_0.shape}")
print(f"         Latents shape: {latents_2_0.shape}")
print(f"         Spectra shape: {spectra_2_0.shape}")
print(f"Unique labels: {np.unique(labels_2_0)}")

# Convert to feather
df_spectra_2_0, path_spectra_2_0 = convert_spectra_to_feather(spectra_2_0, labels_2_0, 2.0)
df_latents_2_0, path_latents_2_0 = convert_latents_to_feather(latents_2_0, labels_2_0, 2.0)

## Verify Output Files

In [ ]:
# Verify each converted file
file_pairs = [
    (1.0, path_spectra_1_0, path_latents_1_0),
    (1.5, path_spectra_1_5, path_latents_1_5),
    (2.0, path_spectra_2_0, path_latents_2_0)
]

for std_val, spectra_path, latents_path in file_pairs:
    print(f"\n{'='*60}")
    print(f"Verifying std {std_val} files")
    print('='*60)
    
    # Verify spectra file
    print(f"\nSPECTRA: {os.path.basename(spectra_path)}")
    df_spec = pd.read_feather(spectra_path)
    print(f"  Shape: {df_spec.shape}")
    print(f"  Columns: {len(df_spec.columns)}")
    print(f"  Has index column: {'index' in df_spec.columns}")
    
    p_cols = [c for c in df_spec.columns if c.startswith('p_')]
    n_cols = [c for c in df_spec.columns if c.startswith('n_')]
    print(f"  P_ columns: {len(p_cols)} (from {p_cols[0]} to {p_cols[-1]})")
    print(f"  N_ columns: {len(n_cols)} (from {n_cols[0]} to {n_cols[-1]})")
    print(f"  Label column: {'Label' in df_spec.columns}")
    print(f"  Label values: {df_spec['Label'].unique()}")
    print(f"  Label counts:\n{df_spec['Label'].value_counts()}")
    
    # Verify latents file
    print(f"\nLATENTS: {os.path.basename(latents_path)}")
    df_lat = pd.read_feather(latents_path)
    print(f"  Shape: {df_lat.shape}")
    print(f"  Columns: {len(df_lat.columns)}")
    print(f"  Has index column: {'index' in df_lat.columns}")
    
    latent_cols = [c for c in df_lat.columns if c.startswith('latent_')]
    print(f"  Latent columns: {len(latent_cols)} (from {latent_cols[0]} to {latent_cols[-1]})")
    print(f"  Label column: {'Label' in df_lat.columns}")
    print(f"  Label values: {df_lat['Label'].unique()}")
    print(f"  Label counts:\n{df_lat['Label'].value_counts()}")
    
    # Verify indices match
    print(f"\n  Index alignment check:")
    print(f"    Spectra index range: {df_spec['index'].min()} to {df_spec['index'].max()}")
    print(f"    Latents index range: {df_lat['index'].min()} to {df_lat['index'].max()}")
    print(f"    Indices match: {df_spec['index'].equals(df_lat['index'])}")

## Compare with Test Data Format

In [ ]:
# Compare format with test_data.feather
print("Comparison between test data and synthetic data (std 1.0):")
print("="*60)

# Get p_ and n_ columns from both
test_p = [c for c in test_df.columns if c.startswith('p_')]
test_n = [c for c in test_df.columns if c.startswith('n_')]
synth_p = [c for c in df_spectra_1_0.columns if c.startswith('p_')]
synth_n = [c for c in df_spectra_1_0.columns if c.startswith('n_')]

print(f"\nSPECTRA FILE:")
print(f"Test data p_ columns: {len(test_p)} ({test_p[0]} to {test_p[-1]})")
print(f"Synthetic p_ columns: {len(synth_p)} ({synth_p[0]} to {synth_p[-1]})")
print(f"Match: {test_p == synth_p}")

print(f"\nTest data n_ columns: {len(test_n)} ({test_n[0]} to {test_n[-1]})")
print(f"Synthetic n_ columns: {len(synth_n)} ({synth_n[0]} to {synth_n[-1]})")
print(f"Match: {test_n == synth_n}")

print(f"\nTest data has Label column: {'Label' in test_df.columns}")
print(f"Synthetic spectra has Label column: {'Label' in df_spectra_1_0.columns}")
print(f"Synthetic latents has Label column: {'Label' in df_latents_1_0.columns}")

print(f"\nTest data Label values: {sorted(test_df['Label'].unique())}")
print(f"Synthetic spectra Label values: {sorted(df_spectra_1_0['Label'].unique())}")
print(f"Synthetic latents Label values: {sorted(df_latents_1_0['Label'].unique())}")

print(f"\nLATENTS FILE:")
print(f"Latent dimensions: {len([c for c in df_latents_1_0.columns if c.startswith('latent_')])}")
print(f"Has Label column: {'Label' in df_latents_1_0.columns}")
print(f"Has index column: {'index' in df_latents_1_0.columns}")

print(f"\nINDEX ALIGNMENT:")
print(f"Spectra has index column: {'index' in df_spectra_1_0.columns}")
print(f"Latents has index column: {'index' in df_latents_1_0.columns}")
print(f"Indices match: {df_spectra_1_0['index'].equals(df_latents_1_0['index'])}")
print(f"\nSample alignment (first 3 rows):")
print(f"Spectra indices: {df_spectra_1_0['index'].head(3).tolist()}")
print(f"Latents indices: {df_latents_1_0['index'].head(3).tolist()}")

print(f"\n✓ Format verification complete!")
print(f"  SPECTRA:")
print(f"    - Index column for alignment: {'index' in df_spectra_1_0.columns}")
print(f"    - Column naming matches: {'p_' and 'n_' prefixes}")
print(f"    - Label column included: {True}")
print(f"    - Labels are strings: {df_spectra_1_0['Label'].dtype == 'object'}")
print(f"  LATENTS:")
print(f"    - Index column for alignment: {'index' in df_latents_1_0.columns}")
print(f"    - Latent columns: {'latent_' prefix}")
print(f"    - Label column included: {True}")
print(f"    - Labels are strings: {df_latents_1_0['Label'].dtype == 'object'}")
print(f"  ALL FILES:")
print(f"    - Index alignment: Spectra and latents can be matched by 'index' column")
print(f"    - File format: .feather")

## Summary

The notebook has successfully converted synthetic data from .npy format to .feather format with:

**Spectra Files:**
- **Index column**: Enables alignment with corresponding latents
- **Proper column headers**: p_ and n_ prefixes with bin numbers (184-1021)
- **Label column**: String labels (DEB, DEM, DMMP, DPM, DtBP, JP8, MES, TEPO) instead of numeric
- **File naming**: synthetic_spectra_std{value}_{labels}.feather

**Latents Files:**
- **Index column**: Enables alignment with corresponding spectra
- **Column headers**: latent_0 through latent_511 (512 dimensions)
- **Label column**: String labels (DEB, DEM, DMMP, DPM, DtBP, JP8, MES, TEPO) instead of numeric
- **File naming**: synthetic_latents_std{value}_{labels}.feather

**Key Feature:**
- The `index` column in both files allows matching which latent generated which spectrum
- Same index value across files = latent → spectrum correspondence

**Both file types:**
- Include label names in filenames
- Match test_data.feather format structure
- Saved in .feather format